# 01 — Exploratory Data Analysis
UCI Heart Disease dataset (UCI ID 45). This notebook documents data quality, distributions, and relationships before any modelling. **Educational use only — not a clinical diagnostic tool.**


## 1. Setup and data loading

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))  # make 'src' importable when run from notebooks/

import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
from src.data import load_dataset

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", None)

RAW = Path.cwd().parent / "data" / "raw" / "heart_disease.csv"
df = load_dataset(RAW)
df.head()

## 2. Dataset overview

In [ ]:
print(f"Rows: {df.shape[0]}, Features: {df.shape[1] - 1}, Target: binary (1 = disease present)")
print("\nDtypes:")
print(df.dtypes)
print("\nDescriptive statistics:")
df.describe().T

## 3. Data quality — missing values and duplicates

In [ ]:
missing = df.isna().sum().rename("missing_count").to_frame()
missing["missing_pct"] = (missing["missing_count"] / len(df) * 100).round(2)
missing = missing[missing["missing_count"] > 0]
print(missing if len(missing) else "No missing values after cleaning.")
print(f"Duplicate rows: {df.duplicated().sum()}")

## 4. Target class balance

In [ ]:
fig, ax = plt.subplots(figsize=(5, 4))
sns.countplot(data=df, x="target", ax=ax)
ax.set_xticklabels(["no disease (0)", "disease (1)"])
ax.set_title("Target class balance")
ax.set_ylabel("count")
for p in ax.patches:
    ax.annotate(f"{int(p.get_height())}", (p.get_x() + p.get_width() / 2, p.get_height()),
                ha="center", va="bottom")
plt.tight_layout()
plt.savefig("../reports/plots/class_balance.png", dpi=150)
plt.show()

## 5. Feature distributions (histograms)

In [ ]:
df.hist(figsize=(14, 10), bins=20, edgecolor="black")
plt.tight_layout()
plt.savefig("../reports/plots/feature_histograms.png", dpi=150)
plt.show()

## 6. Correlation analysis

In [ ]:
plt.figure(figsize=(10, 8))
sns.heatmap(df.corr(numeric_only=True), annot=True, fmt=".2f", cmap="coolwarm", center=0)
plt.title("Feature correlation heatmap")
plt.tight_layout()
plt.savefig("../reports/plots/correlation_heatmap.png", dpi=150)
plt.show()

## 7. Key clinical features vs target

In [ ]:
key_features = ["age", "trestbps", "chol", "thalach", "oldpeak"]
fig, axes = plt.subplots(1, len(key_features), figsize=(18, 4))
for ax, feat in zip(axes, key_features):
    sns.boxplot(data=df, x="target", y=feat, ax=ax)
    ax.set_title(feat)
plt.tight_layout()
plt.savefig("../reports/plots/feature_vs_target_boxplots.png", dpi=150)
plt.show()

## 8. Observations (personalize with your own findings)
- **Class balance:** roughly balanced binary target — no resampling required.
- **`thalach` (max heart rate)** shows the strongest negative correlation with the target.
- **`cp` (chest pain type)** and **`exang`** correlate positively with disease presence.
- `age`, `trestbps`, `chol` are unimodal with mild right skew — StandardScaler is appropriate.
- Categorical codes (`sex`, `cp`, `fbs`, `restecg`, `exang`, `slope`, `ca`, `thal`) are one-hot encoded in the preprocessing pipeline.
- Correlations are associations only — no causal claims are made.
